### Import & Load Data

In [1]:
import pandas as pd
import numpy as np
import re
import time
import multiprocessing
from multiprocessing.pool import ThreadPool

print("Loading dataset...")
df = pd.read_csv('../../data_resource/recipes.csv')
print(f"Dataset loaded: {len(df)} rows.")

Loading dataset...
Dataset loaded: 522517 rows.


### Preprocessing Function

In [2]:
def preprocess_data(df_chunk):
    for col in ['RecipeIngredientParts', 'RecipeInstructions']:
        df_chunk[col] = (df_chunk[col].fillna('')
                                      .astype(str)
                                      .str.replace(r'^c\((.*)\)$', r'\1', regex=True)
                                      .str.replace('character(0)', '', regex=False))

    df_chunk['Name'] = df_chunk['Name'].fillna('')

    combined = df_chunk['Name'] + " " + df_chunk['RecipeIngredientParts'] + " " + df_chunk['RecipeInstructions']

    df_chunk['clean_text'] = combined.str.lower().apply(lambda x: re.sub(r'[^\w\s]', '', x))

    return df_chunk

### Multiprocessing Benchmark

In [3]:
print("Starting preprocessing benchmark...")

c = multiprocessing.cpu_count()
print(f"Maximum cores available: {c}")

df_split = np.array_split(df, c)

print("Running with 1 core...")
start_1 = time.time()
with ThreadPool(1) as pool:
    result_1 = pool.map(preprocess_data, df_split)
time_1 = time.time() - start_1
print(f"Execution time (1 core): {time_1:.4f} s")

print(f"Running with {c} cores...")
start_n = time.time()
with ThreadPool(c) as pool:
    result_n = pool.map(preprocess_data, df_split)
time_n = time.time() - start_n
print(f"Execution time ({c} cores): {time_n:.4f} s")

speedup = time_1 / time_n
print(f"Speedup: {speedup:.2f}x")

print("Saving preprocessed data...")
processed_df = pd.concat(result_n)
processed_df[['RecipeId', 'Name', 'clean_text']].to_csv('../../data_resource/preprocessed_recipes.csv', index=False)
print("Process completed.")

Starting preprocessing benchmark...
Maximum cores available: 12


C:\Users\Nitro V15\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Running with 1 core...
Execution time (1 core): 11.3695 s
Running with 12 cores...
Execution time (12 cores): 11.1895 s
Speedup: 1.02x
Saving preprocessed data...
Process completed.
